In [1]:
import os 
os.environ['HTTPS_PROXY']='czgdcsdwankerb1.czds.bz:8080'
#!pip install openai
#!pip install openai
#!pip install python-Levenshtein

In [2]:
import pandas as pd

In [3]:
units_from_the_db = pd.read_csv(r"../Data/PROPERTY.csv")['UNIT_OF_MEAS_SI'].dropna().unique().tolist()
units_from_the_db

['°C',
 'MPa',
 'kg/m³',
 '%',
 '-',
 'kJ/m²',
 'class',
 'E-4/°C',
 'Ohm*m',
 'E-4',
 'g/cm³',
 'ml/g',
 'cm³/g',
 'M-Scale',
 'mm',
 'Ohm',
 'kN/m',
 'µm',
 'g/10min',
 'kV/mm',
 'W/(m K)',
 'cm³/10min',
 '% by wt.',
 'GPa',
 'Pa s',
 'dB',
 'g/mol',
 'mm²/s',
 'g/m²',
 's',
 'Ohm*cm',
 'J/m',
 'mm/min',
 'J/(kg K)',
 'µgC/g',
 'µg/g',
 '% by vol.',
 'R-Scale',
 'min',
 'J']

In [4]:
tds_units = pd.read_csv(r"../Data/TDS_MAPPING.csv")['TDS_UNIT'].dropna().unique().tolist()
tds_units

['%',
 'cm3/g',
 'mm3',
 'U050=100',
 'Index',
 'ppm',
 'meq/kg',
 '°C',
 'min/mm',
 'g/cm3',
 'g/cm³',
 'sec.',
 'sec',
 'S',
 's',
 '-',
 '106 g/mol',
 'g / mol',
 'g/mol',
 '10 g/mol 6',
 'um',
 'u m',
 'micron',
 'µm',
 'J/(kg K)',
 '㎛',
 'psi',
 'MPa',
 'bar',
 'N/mm2',
 '×10 -3m',
 'n°/kg',
 'kg/m3',
 'Kg/lit',
 'g/cm 3',
 'mm',
 'class',
 'Class',
 'mm/min',
 'cm3/(m2*d*bar)',
 'KJ/m2',
 'kJ/m²',
 'kJ/m2',
 'kj/m 2',
 'kj/m2',
 'KJ/m 2',
 'kJ/m 2',
 'Kj/m2',
 '(kJ/m²)',
 'ft*lb/in2',
 'kJ/mÂ²',
 'KJ/m²',
 'ISO 179/1eA',
 'ft·lb/in²',
 '°c',
 'kN/cmÂ²',
 'cm/cm/°C',
 'cm/cm/C',
 'in^-5/in/°F',
 'E-6/K',
 '10^-6/°C',
 '10-4/°C',
 '×10 -5 /K',
 'Ã—10-5 /°C',
 'Ã—10-5/°C',
 'Ã—105/°C',
 'x105/°C',
 'E-4/°C',
 'E-4/ °c',
 'µm/mK',
 'x 10-5 /K',
 'Static',
 'Dynamic',
 'K –1',
 '× E-6/K',
 '/°C',
 'in/in/°F x10-4',
 '10-6 / K',
 '10 -6 /°C',
 'µm/(m.K)',
 '×10-5/℃',
 'x10',
 'x107',
 'x10-',
 'x10-5/℃',
 '10 /K',
 'm/(m.K)',
 'Unit length / °F',
 'm/mK',
 'V',
 'PLC',
 'Volt',
 'Ratin

### Unit conversion shared by Abhishek

In [5]:
conversion_table1 = pd.read_excel("Unit_conversions.xlsx")
conversion_table1 = conversion_table1[conversion_table1['Incoming Unit']!="NULL"]
conversion_table1 = conversion_table1[conversion_table1['Incoming Unit']!="-"]
conversion_table1.dropna(inplace=True)
conversion_table1.reset_index(drop=True,inplace=True)
conversion_table1

,Incoming Unit,ceed_unit,formula
0,GPA,MPa,(x)*1000
1,IN/IN,%,(x)*100
2,PSI,MPa,(x)*0.00689476
3,ºF,°C,((x) - 32)*(5/9)
4,G/CM3,kg/m³,(x)*1000
5,G/CM³,kg/m³,(x)*1000
6,8/CM3,kg/m³,(x)*1000
7,KG/M3,g/cm³,(x)*0.001
8,FT-LB/IN2,kJ/m²,(x)*2.10152
9,FT.LB/IN2,kJ/m²,(x)*2.10152


In [6]:
additional_exception_units = pd.DataFrame()

additional_exception_units = pd.concat([additional_exception_units, conversion_table1[conversion_table1['ceed_unit']=="g/cm³"]], axis=0)
additional_exception_units = pd.concat([additional_exception_units, conversion_table1[conversion_table1['ceed_unit']=="µm"]], axis=0)

conversion_table1 = conversion_table1[conversion_table1['ceed_unit']!="g/cm³"]
conversion_table1 = conversion_table1[conversion_table1['ceed_unit']!="µm"]

additional_exception_units

,Incoming Unit,ceed_unit,formula
7,KG/M3,g/cm³,(x)*0.001
17,UM,µm,x


In [7]:
conversion_table1['ceed_unit'].unique()

array(['MPa', '%', '°C', 'kg/m³', 'kJ/m²', 'J/m', 'E-4/°C', 'E-4',
       'Ohm*m', 'E-6'], dtype=object)

### Additional conversion formula created using GPT

In [8]:
conversion_table2 = pd.read_csv("conversion_table.csv")
conversion_table2 = conversion_table2[conversion_table2['ceed_unit']!='Ohm*m']
additional_units = {"Incoming Unit":['um'],
                "ceed_unit":['mm'],
                "formula":['x/1000']}
conversion_table2 = pd.concat([conversion_table2, pd.DataFrame(additional_units)], axis=0)

In [9]:
conversion_table2['ceed_unit'].unique()

array(['Mpa', 'kg/m³', 'kJ/m²', 'ml/g', 'cm³/g', 'mm', 'Ohm', 'kN/m',
       'g/10min', 'kV/mm', 'W/(m K)', 'cm³/10min', 'Pa s', 'dB', 'g/mol',
       'mm²/s', 'g/m²', 's', 'J/m', 'mm/min', 'J/(kg K)', 'µgC/g', 'µg/g',
       'J', '°C', 'MPa', 'g/cm³', 'Ohm*cm'], dtype=object)

In [10]:
conversion_table3 = pd.read_csv("conversion_table2.csv")
conversion_table3 = conversion_table3[:286]

#adding ohm*m
conversion_table2 = pd.concat([conversion_table2, conversion_table3[conversion_table3['ceed_unit']=='Ohm*m']], axis=0)
conversion_table2=conversion_table2[conversion_table2['ceed_unit']!='Ohm*cm']
conversion_table2.reset_index(drop=True,inplace=True)

In [11]:
final_unit_conversion_table = pd.concat([conversion_table1,conversion_table2],axis=0).drop_duplicates().reset_index(drop=True)
final_unit_conversion_table['Incoming Unit'] = final_unit_conversion_table['Incoming Unit'].apply(lambda x: ''.join(x.lower().split()))
final_unit_conversion_table.drop_duplicates(inplace=True)




additional_units_for_ohm_m = {"Incoming Unit":['ohm-m', 'ohm.m','ohm m', 'ohmm'],
                "ceed_unit":['Ohm*m','Ohm*m','Ohm*m','Ohm*m'],
                "formula":['x','x','x','x']}
final_unit_conversion_table = pd.concat([final_unit_conversion_table, pd.DataFrame(additional_units_for_ohm_m)], axis=0)

final_unit_conversion_table.reset_index(drop=True,inplace=True)
final_unit_conversion_table.to_csv("../dependencies/final_unit_conversion_table.csv",index=False)

final_unit_conversion_table

,Incoming Unit,ceed_unit,formula
0,gpa,MPa,(x)*1000
1,in/in,%,(x)*100
2,psi,MPa,(x)*0.00689476
3,ºf,°C,((x) - 32)*(5/9)
4,g/cm3,kg/m³,(x)*1000
...,...,...,...
1474,yohm*m,Ohm*m,x/1e24
1475,ohm-m,Ohm*m,x
1476,ohm.m,Ohm*m,x
1477,ohm m,Ohm*m,x


In [12]:
final_unit_conversion_table['ceed_unit'].unique()

array(['MPa', '%', '°C', 'kg/m³', 'kJ/m²', 'J/m', 'E-4/°C', 'E-4',
       'Ohm*m', 'E-6', 'Mpa', 'ml/g', 'cm³/g', 'mm', 'Ohm', 'kN/m',
       'g/10min', 'kV/mm', 'W/(m K)', 'cm³/10min', 'Pa s', 'dB', 'g/mol',
       'mm²/s', 'g/m²', 's', 'mm/min', 'J/(kg K)', 'µgC/g', 'µg/g', 'J',
       'g/cm³'], dtype=object)

### conversion folrmula for exceptions

In [13]:
final_unit_conversion_table_for_exeptions = pd.read_csv("conversion_table3.csv")

In [15]:
actual_units = {"Incoming Unit":['Gpa','Ohm*cm','μm','um','min'],
                "ceed_unit":['Gpa','Ohm*cm','μm','μm','min'],
                "formula":['x','x','x','x','x']}
final_unit_conversion_table_for_exeptions = pd.concat([final_unit_conversion_table_for_exeptions, pd.DataFrame(actual_units)], axis=0)
final_unit_conversion_table_for_exeptions.reset_index(drop=True,inplace=True)

In [16]:
final_unit_conversion_table_for_exeptions

,Incoming Unit,ceed_unit,formula
0,mOhm*cm,Ohm*cm,x/1000
1,milliohm centimeter,Ohm*cm,x/1000
2,kOhm*cm,Ohm*cm,x*1000
3,kiloohm centimeter,Ohm*cm,x*1000
4,MOhm*cm,Ohm*cm,x*1000000
...,...,...,...
276,Gpa,Gpa,x
277,Ohm*cm,Ohm*cm,x
278,μm,μm,x
279,um,μm,x


In [17]:
final_unit_conversion_table_for_exeptions = pd.concat([final_unit_conversion_table_for_exeptions, additional_exception_units], axis=0)

additional_units_for_ohm_cm = {"Incoming Unit":['ohm-cm', 'ohm.cm','ohm cm','ohmcm'],
                "ceed_unit":['Ohm*cm','Ohm*cm','Ohm*cm','Ohm*cm'],
                "formula":['x','x','x','x']}
final_unit_conversion_table_for_exeptions = pd.concat([final_unit_conversion_table_for_exeptions, pd.DataFrame(additional_units_for_ohm_cm)], axis=0)

final_unit_conversion_table_for_exeptions['Incoming Unit'] = final_unit_conversion_table_for_exeptions['Incoming Unit'].apply(lambda x: ''.join(x.lower().split()))
final_unit_conversion_table_for_exeptions.drop_duplicates(inplace=True)
final_unit_conversion_table_for_exeptions
final_unit_conversion_table_for_exeptions.to_csv("../dependencies/final_unit_conversion_table_for_exeptions.csv",index=False)

In [18]:
final_unit_conversion_table_for_exeptions['ceed_unit'].unique()

array(['Ohm*cm', 'GPa', 'g/cm³', 'µm', 'min', 'Gpa', 'μm'], dtype=object)